In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

from matplotlib import pyplot as plt
from matplotlib_venn import venn3

import statsmodels.api as sm

import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import descri_function as des_fun

In [15]:
# Data with maing models

#df = pd.read_parquet("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_27_03_2026_with_MEM_EVI.parquet")
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_outbreak_mmaing/mmaing_atend_ivas.parquet')
#df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_with_MEM_26_03_2026_mmaingpar.parquet')

# data with ears and evi model

df2 = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_06_05_2026_with_MEM_ears_EVI_timing.parquet') 

dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')


In [3]:
#df["year_week"] = df["ano"].astype(int).astype(str) + "-" + df["epiweek"].astype(str).str.zfill(2)

#df = df.rename(columns = {'EWS_ISF_atend_ivas':'EWS_ISF',
# 'EWS_LOF_atend_ivas':'EWS_LOF',
#'EWS_OCSVM_atend_ivas':'EWS_OCSVM',
# 'EWS_COPOD_atend_ivas':'EWS_COPOD',
# 'EWS_Rt_atend_ivas': 'EWS_Rt'})

In [4]:
#df.columns.to_list()

In [16]:
df = df[['co_ibge',  'EWS_ISF', 'EWS_LOF', 'EWS_OCSVM', 'EWS_COPOD', 'EWS_Rt', 'year_week']]

In [17]:
# Select cities for the manuscript analysis (valid MEM)
lst = list(set(df.co_ibge.unique()) - set(dta.co_ibge.unique()))
df = df[~df.co_ibge.isin(lst)]

df =  df[(df.year_week >= '2022-42') &(df.year_week <= '2025-32')]

In [18]:
df = df.replace({'Não': 0, 'Sim': 1})

In [8]:
df.columns

Index(['co_ibge', 'EWS_ISF', 'EWS_LOF', 'EWS_OCSVM', 'EWS_COPOD', 'EWS_Rt',
       'year_week'],
      dtype='object')

In [9]:
df2.columns

Index(['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', 'sinal_ears_atend', 'sinal_evi_ivas'],
      dtype='object')

In [10]:
df_merged = df2.merge(df, on=['co_ibge', 'year_week'], how='inner')

In [11]:
df_merged

,co_ibge,epiweek,year,year_week,week,atend_ivas,atend_totais,mem_surge_01_correct_with_consec,warning_final_mem_surge_01,sinal_ears_atend,sinal_evi_ivas,EWS_ISF,EWS_LOF,EWS_OCSVM,EWS_COPOD,EWS_Rt
0,110001,42,2022.0,2022-42,2022-10-23,31,648,0,0,1.0,0,1,1,1,0,1
1,110001,43,2022.0,2022-43,2022-10-30,16,650,0,0,0.0,0,0,0,0,0,0
2,110001,44,2022.0,2022-44,2022-11-06,20,558,0,0,0.0,0,0,0,0,0,0
3,110001,45,2022.0,2022-45,2022-11-13,32,677,0,0,1.0,0,1,1,1,0,1
4,110001,46,2022.0,2022-46,2022-11-20,47,808,0,0,1.0,1,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788650,530010,28,2025.0,2025-28,2025-07-13,9099,73467,0,0,0.0,0,0,0,0,0,0
788651,530010,29,2025.0,2025-29,2025-07-20,8762,73637,0,0,0.0,0,0,0,0,0,0
788652,530010,30,2025.0,2025-30,2025-07-27,7914,73686,0,0,0.0,0,0,0,0,0,0
788653,530010,31,2025.0,2025-31,2025-08-03,7146,79785,0,0,0.0,0,0,0,0,0,0


In [12]:
df_merged.columns

Index(['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', 'sinal_ears_atend', 'sinal_evi_ivas',
       'EWS_ISF', 'EWS_LOF', 'EWS_OCSVM', 'EWS_COPOD', 'EWS_Rt'],
      dtype='object')

In [13]:
df_merged.warning_final_mem_surge_01.sum()

26411

In [14]:
from pathlib import Path
from datetime import datetime

out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")

fname = f"aesop_{datetime.now():%d_%m_%Y}_with_MEM_all_models.parquet"

df_merged.to_parquet(out_dir / fname)